In [ ]:
from pyspark.sql import SparkSession

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving most_subscribed_youtube_channels.csv to most_subscribed_youtube_channels.csv


# **ACTIVITY #3 - Distributed Computing Services**
Members:
- Rico Cadag
- Julie Ana Merle
-  Mikaela Francine Miranda


In [ ]:
import csv
from io import StringIO
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Activity_Lecture4").getOrCreate()
sc = spark.sparkContext

# 1. LOAD DATASET
data = sc.textFile("/content/most_subscribed_youtube_channels.csv")

# Remove header
header = data.first()
data = data.filter(lambda line: line != header)

print("Initial partitions:", data.getNumPartitions())

# Define a function to parse each CSV line using the csv module
def parse_csv_line(line):
    if not line.strip(): # Skip empty lines
        return []
    # Use StringIO to treat the string as a file for csv.reader
    reader = csv.reader(StringIO(line))
    try:
        return next(reader)
    except StopIteration:
        # Handle cases where a line might be malformed or empty after stripping
        return []

# 2. SPLIT CSV ROWS using the robust parser
rows = data.map(parse_csv_line).filter(lambda x: len(x) > 0) # Filter out empty/malformed rows


# ------------------------------------------------
# PARTITION STRATEGY 1: HASH PARTITIONING
# ------------------------------------------------

# Key = category
category_pairs = rows.map(lambda x: (x[5], 1))

# Apply hash partitioning
category_partitioned = category_pairs.partitionBy(8)

print("Hash partitions:", category_partitioned.getNumPartitions())


# Count channels per category
category_counts = category_partitioned.reduceByKey(lambda a, b: a + b)

print("\nTOP CATEGORIES")
top_categories = category_counts.sortBy(lambda x: x[1], ascending=False).take(10)

for cat, count in top_categories:
    print(cat, "->", count)


# Filter example
popular_categories = category_counts.filter(lambda x: x[1] >= 50)

print("\nCATEGORIES WITH >= 50 CHANNELS")
for cat, count in popular_categories.collect():
    print(cat, "->", count)



# ------------------------------------------------
# PARTITION STRATEGY 2: RANGE PARTITIONING
# ------------------------------------------------

# Key = subscribers
# Ensure the int conversion is robust for potentially messy data in the subscribers column
def get_subscribers(x):
    try:
        return int(x[2].replace(",", "").strip('"'))
    except (ValueError, IndexError):
        # Return None for rows where subscribers cannot be parsed
        return None

subscriber_pairs = rows.map(lambda x: (get_subscribers(x), (x[1], x[5], x[6]))).filter(lambda x: x[0] is not None)

# Range partitioning happens with sortByKey
sorted_subscribers = subscriber_pairs.sortByKey(ascending=False, numPartitions=8)

print("\nRange partitions:", sorted_subscribers.getNumPartitions())


print("\nTOP 10 CHANNELS BY SUBSCRIBERS")
for subs, info in sorted_subscribers.take(10):
    name, category, started = info
    print(name, "|", category, "| started", started, "| subscribers", subs)


# Filter example
recent_channels = sorted_subscribers.filter(lambda x: int(x[1][2]) >= 2015)

print("\nCHANNELS STARTED 2015+")
for subs, info in recent_channels.take(10):
    name, category, started = info
    print(name, "| started", started, "| subscribers", subs)


spark.stop()

Initial partitions: 2
Hash partitions: 8

TOP CATEGORIES
Entertainment -> 241
Music -> 222
People & Blogs -> 119
Gaming -> 102
Comedy -> 63
Film & Animation -> 52
Education -> 46
Howto & Style -> 45
 -> 27
News & Politics -> 27

CATEGORIES WITH >= 50 CHANNELS
Film & Animation -> 52
Gaming -> 102
People & Blogs -> 119
Comedy -> 63
Music -> 222
Entertainment -> 241

Range partitions: 8

TOP 10 CHANNELS BY SUBSCRIBERS
T-Series | Music | started 2006 | subscribers 222000000
YouTube Movies | Film & Animation | started 2015 | subscribers 154000000
Cocomelon - Nursery Rhymes | Education | started 2006 | subscribers 140000000
SET India | Shows | started 2006 | subscribers 139000000
Music |  | started 2013 | subscribers 116000000
PewDiePie | Gaming | started 2010 | subscribers 111000000
MrBeast | Entertainment | started 2012 | subscribers 102000000
✿ Kids Diana Show | People & Blogs | started 2015 | subscribers 99700000
Like Nastya | People & Blogs | started 2016 | subscribers 99200000
Gaming |

# **ACTIVITY #4 - DATA PRE-PROCESSING**


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import col, desc

spark = SparkSession.builder \
    .appName("YouTube_Data_Preprocessing") \
    .getOrCreate()


# SCHEMA (Lecture 5 requirement)
schema = StructType([
    StructField("rank", IntegerType(), True),
    StructField("Youtuber", StringType(), True),
    StructField("subscribers", StringType(), True),
    StructField("video_views", StringType(), True),
    StructField("video_count", StringType(), True),
    StructField("category", StringType(), True),
    StructField("started", IntegerType(), True)
])


# LOAD CSV AS DATAFRAME
df = spark.read.csv(
    "most_subscribed_youtube_channels.csv",
    header=True,
    schema=schema
)

print("ORIGINAL DATA")
df.show(5)



# DATA PREPROCESSING
# --------------------

# 1 Remove missing values
clean_df = df.dropna()

# 2 Remove duplicate rows
clean_df = clean_df.dropDuplicates()

print("\nCLEANED DATA")
clean_df.show(5)


# -------------------------------------------------
# INSIGHT 1
# Top 5 categories with most channels
# -------------------------------------------------
print("\nTOP CATEGORIES")

top_categories = clean_df.groupBy("category") \
    .count() \
    .orderBy(desc("count"))

top_categories.show(5)


# -------------------------------------------------
# INSIGHT 2
# Top 10 most subscribed channels
# -------------------------------------------------
print("\nTOP SUBSCRIBED CHANNELS")

top_subscribers = clean_df.orderBy(desc("subscribers"))

top_subscribers.show(10)


# -------------------------------------------------
# INSIGHT 3
# Channels created after 2015
# -------------------------------------------------
print("\nCHANNELS STARTED AFTER 2015")

recent_channels = clean_df.filter(col("started") >= 2015)

recent_channels.show(10)


# -------------------------------------------------
# SAVE RESULTS AS PARQUET
# -------------------------------------------------
clean_df.write.format("parquet") \
    .mode("overwrite") \
    .save("youtube_cleaned_parquet")


spark.stop()

ORIGINAL DATA
+----+--------------------+-----------+---------------+-----------+----------------+-------+
|rank|            Youtuber|subscribers|    video_views|video_count|        category|started|
+----+--------------------+-----------+---------------+-----------+----------------+-------+
|   1|            T-Series|222,000,000|198,459,090,822|     17,317|           Music|   2006|
|   2|      YouTube Movies|154,000,000|              0|          0|Film & Animation|   2015|
|   3|Cocomelon - Nurse...|140,000,000|135,481,339,848|        786|       Education|   2006|
|   4|           SET India|139,000,000|125,764,252,686|     91,271|           Shows|   2006|
|   5|               Music|116,000,000|              0|          0|            NULL|   2013|
+----+--------------------+-----------+---------------+-----------+----------------+-------+
only showing top 5 rows

CLEANED DATA
+----+------------------+-----------+--------------+-----------+-------------+-------+
|rank|          Youtuber